In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import Subset, DataLoader, TensorDataset

def stratified_split_on_A(dataset: TensorDataset, A_tensor, train_frac=0.8, n_bins=10, seed=42):
    """
    dataset: TensorDataset(Y_t, A_t, Z_t)  (or similar)
    A_tensor: torch.Tensor or np.array of shape (N,) with treatment values
    """
    if hasattr(A_tensor, "numpy"):
        A = A_tensor.cpu().numpy().ravel()
    else:
        A = np.array(A_tensor).ravel()

    quantiles = np.linspace(0, 100, n_bins + 1)
    bins = np.nanpercentile(A, quantiles)
    A_bins = np.digitize(A, bins[1:-1], right=True)

    sss = StratifiedShuffleSplit(n_splits=1, train_size=train_frac, random_state=seed)
    indices = np.arange(len(A))
    train_idx, valid_idx = next(sss.split(indices, A_bins))

    train_ds = Subset(dataset, train_idx)
    valid_ds = Subset(dataset, valid_idx)
    return train_ds, valid_ds, train_idx, valid_idx

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, TensorDataset, random_split

seed = 0
train_all = True

def generate_synthetic_low_dim(n):
    # U1 ~ U[-1, 2]
    U1 = np.random.uniform(-1, 2, n)
    
    # U2 ~ U[0, 1] - 1[0 <= U1 <= 1]
    indicator = ((U1 >= 0) & (U1 <= 1)).astype(float)
    U2 = np.random.uniform(0, 1, n) - indicator
    
    # W = [U2 + U[-1, 1], U1 + N(0, 1)]
    W1 = U2 + np.random.uniform(-1, 1, n)
    W2 = U1 + np.random.normal(0, 1, n)
    W = np.stack([W1, W2], axis=1) # Shape [n, 2]
    
    # Z = [U2 + N(0, 1), U1 + U[-1, 1]]
    Z1 = U2 + np.random.normal(0, 1, n)
    Z2 = U1 + np.random.uniform(-1, 1, n)
    Z = np.stack([Z1, Z2], axis=1) # Shape [n, 2]
    
    # A := U1 + N(0, 1)
    A = U1 + np.random.normal(0, 1, n)
    
    # Y := 3 cos(2(0.3U2 + 0.3U1 + 0.2) + 1.5A) + N(0, 1)
    inner_term = 2 * (0.3 * U2 + 0.3 * U1 + 0.2) + 1.5 * A
    Y = 3 * np.cos(inner_term) + np.random.normal(0, 1, n)
    
    return {
        'W': W,
        'Z': Z,
        'A': A,
        'Y': Y
    }

sample_size = 500

set_seed(seed)
data = generate_synthetic_low_dim(sample_size)
cuda_available = torch.cuda.is_available()
device = torch.device("cuda:3" if cuda_available else "cpu")

In [ ]:
# ————— Prepare dataset for p(W | A, Z) —————
set_seed(seed)

# Ensure tensors are prepared from the synthetic data generation
# A: [N, 1], Z: [N, 2], W: [N, 2]
A_tensor = torch.tensor(data['A'], dtype=torch.float32).unsqueeze(1)
Z_tensor = torch.tensor(data['Z'], dtype=torch.float32)
W_tensor = torch.tensor(data['W'], dtype=torch.float32)

# Input conditions: [A, Z1, Z2] -> shape [N, 3]
conditions = torch.cat([A_tensor, Z_tensor], dim=1) 
# Target: [W1, W2] -> shape [N, 2]
targets = W_tensor

dataset = TensorDataset(conditions, targets)

# --- NOTE: no validation/test split here; use whole dataset for training ---
batch_size = 64
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ————— Define model p(W | A, Z) —————
class W_given_AZ(nn.Module):
    def __init__(self):
        super(W_given_AZ, self).__init__()
        self.base_net = nn.Sequential(
            nn.Linear(3, 16), # Input: A (1) + Z (2) = 3
            nn.ReLU()
        )
        # Output is now 2D for the two components of W
        self.mean_net = nn.Linear(16, 2)
        self.std_net = nn.Sequential(
            nn.Linear(16, 2),
            nn.Softplus()
        )

    def forward(self, conds):
        features = self.base_net(conds)
        mean = self.mean_net(features)
        std = self.std_net(features) + 1e-6 # Numerical stability
        return mean, std

model = W_given_AZ().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 500

# Track best model by train loss (since no validation set)
best_train_loss = float('inf')

if train_all:
    print("Starting training for p(W|A,Z) on the entire dataset (no val set)...")
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for conds, real_W in train_dataloader:
            conds, real_W = conds.to(device), real_W.to(device)
            optimizer.zero_grad()
            mean, std = model(conds)
            
            # W is 2D -> diagonal Gaussian
            dist = Normal(mean, std)
            # Negative log-likelihood: sum across W dims, mean across batch
            loss = -dist.log_prob(real_W).sum(dim=1).mean()
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * conds.size(0)  # accumulate weighted by batch size

        # Compute average train loss across all samples
        avg_train_loss = train_loss / len(dataset)

        # Print occasionally
        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Train NLL: {avg_train_loss:.6f}')

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

set_seed(seed)

train_all = False

Y_tensor = torch.tensor(data['Y'], dtype=torch.float32).unsqueeze(1)
A_tensor = torch.tensor(data['A'], dtype=torch.float32).unsqueeze(1)
Z_tensor = torch.tensor(data['Z'], dtype=torch.float32)
W_tensor = torch.tensor(data['W'], dtype=torch.float32)

dataset = TensorDataset(Y_tensor, A_tensor, Z_tensor, W_tensor)

batch_size = 64

# NOTE: use full dataset for training (no validation split)
train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

W_model = model
W_model.eval()

    
class NN_TW(nn.Module):
    def __init__(self):
        super(NN_TW, self).__init__()
        # Input: T_rep (u_dim) + U (u_dim) + W (2D) = 2 + 2*u_dim
        self.fc = nn.Sequential(
            nn.Linear(1 + 2, 32),
            # nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, t_rep, w):
        x = torch.cat((t_rep, w), dim=1)
        return self.fc(x)

# 4. Initialize and Optimizer
nn_tw = NN_TW().to(device)


optimizer_adam = torch.optim.AdamW([
    {'params': nn_tw.parameters(), 'lr': 1e-2},
], weight_decay = 1e-4)


mse_loss = nn.MSELoss()
num_epochs = 1000

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_adam,
    T_max=num_epochs,   # total number of epochs
    eta_min=1e-6        # minimum LR
)

# Track best model by train loss (since no validation set)
best_train_loss = float('inf')

# Pre-set sampling params (use consistent values each epoch)
num_samples = 100
eps_per_w = 5

# 5. Training Loop (train on entire dataset and select best model by train loss)
if train_all:
    for epoch in range(num_epochs):
        nn_tw.train()
        train_loss_accum = 0.0  # sum of sample losses (not batch means)
        total_samples_trained = 0

        for y_batch, a_batch, z_batch, w_batch in train_dataloader:
            B = y_batch.shape[0]
            y_batch, a_batch = y_batch.to(device), a_batch.to(device)
            z_batch, w_batch = z_batch.to(device), w_batch.to(device)

            # Sample latent proxies from p(W | A, Z)
            AZ_conds = torch.cat([a_batch, z_batch], dim=1) # [B, 3]
            mean, std = W_model(AZ_conds)
            w_dist = torch.distributions.Normal(mean, std)
            # w_samps shape: [B, num_samples, 2]
            w_samps = w_dist.sample((num_samples,)).permute(1, 0, 2).detach()

            # Reshape for sequential processing
            # [B * num_samples, 2] then repeat_interleave eps_per_w -> [B*num_samples*eps_per_w, 2]
            w_samps_rep = w_samps.reshape(B * num_samples, 2)
            a_rep = a_batch.repeat_interleave(num_samples, dim=0)

            bridge_y = nn_tw(a_rep, w_samps_rep)
            bridge_y = bridge_y.view(B, num_samples, 1).mean(dim=1)

            l1 = mse_loss(bridge_y, y_batch)

            total_loss = l1

            optimizer_adam.zero_grad()
            total_loss.backward()
            optimizer_adam.step()

            # accumulate sample-weighted loss
            train_loss_accum += total_loss.item() * B
            total_samples_trained += B

        # compute exact average train loss per sample
        avg_train_loss = train_loss_accum / total_samples_trained

        scheduler.step()

        # print progress occasionally
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Avg Train Loss: {avg_train_loss:.6f}')

        # # save models when train loss improves
        if avg_train_loss < best_train_loss:
            best_train_loss = avg_train_loss
            torch.save(nn_tw.state_dict(), f'lowD2_nn_tw_{sample_size}_{seed}_adv.pth')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

set_seed(seed)

# 1. Ground Truth ATE (remains the same)
def get_synthetic_ground_truth(a_val, num_samples=100000):
    u1 = np.random.uniform(-1, 2, num_samples)
    u2 = np.random.uniform(0, 1, num_samples) - ((u1 >= 0) & (u1 <= 1)).astype(float)
    inner = 2 * (0.3 * u2 + 0.3 * u1 + 0.2) + 1.5 * a_val
    return np.mean(3 * np.cos(inner))

# 2. Load and Eval setup
nn_tw.load_state_dict(torch.load(f'lowD2_nn_tw_{sample_size}_{seed}_adv.pth', map_location=device))
nn_tw.eval()

A_tests = np.linspace(-1.0, 2.0, 100) 
ob_results_pred = []
results_true = []
mse_sum = 0

current_N = W_tensor.shape[0]
W_obs = W_tensor.to(device) 

print(f"{'A':>6} | {'True ATE':>10} | {'Pred ATE':>10} | {'MSE':>10}")
print("-" * 45)

for a_val in A_tests:
    ate_true = get_synthetic_ground_truth(a_val)
    results_true.append(ate_true)
    
    with torch.no_grad():
        a_tensor = torch.full((current_N, 1), a_val, dtype=torch.float32, device=device)
        
        # Predict in SCALED space
        bridge_y_scaled = nn_tw(a_tensor, W_obs)
        ate_pred_scaled = torch.mean(bridge_y_scaled).item()

        ate_pred = ate_pred_scaled
        ob_results_pred.append(ate_pred)
        
    mse_cur = (ate_pred - ate_true) ** 2
    mse_sum += mse_cur

    print(f"{a_val:6.2f} | {ate_true:10.6f} | {ate_pred:10.6f} | {mse_cur:10.8f}")

mse_avg = mse_sum / len(A_tests)
print("-" * 45)
print(f"Average MSE: {mse_avg:.8f}")

# 5. Optional: Plotting results
plt.figure(figsize=(8, 5))
plt.plot(A_tests, results_true, 'k--', label='True ATE (do-calculus)')
plt.plot(A_tests, ob_results_pred, 'r-', label='Predicted ATE (Bridge)')
plt.xlabel('Treatment A')
plt.ylabel('Outcome Y')
plt.title('Synthetic Low Dimensional: Treatment Effect Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

ob_result_pred = ob_results_pred

print(f"Average MSE: {mse_sum / len(A_tests):.8f}")

# Plotting... (same as your provided plotting code)

In [ ]:
# For p(A|W) in Synthetic Low Dimensional Experiment:
# W = [U2 + U[-1, 1], U1 + N(0, 1)] (shape [N, 2])
# A = U1 + N(0, 1)                  (shape [N, 1])

# 1. Prepare Tensors
# Assuming 'data' comes from generate_synthetic_low_dim(sample_size)
W_tensor = torch.tensor(data['W'], dtype=torch.float32)               # shape [N, 2]
A_tensor = torch.tensor(data['A'], dtype=torch.float32).unsqueeze(1) # shape [N, 1]
Z_tensor = torch.tensor(data['Z'], dtype=torch.float32)               # shape [N, 2]

# Create dataset for p(A|W)
conds_A = torch.concat([W_tensor, Z_tensor], dim=1)                     # W (Input, now 2D)
targets_A = A_tensor                   # A (Target, scalar)
dataset_A = TensorDataset(conds_A, targets_A)

set_seed(seed)

# Using your batch size of 16 as per previous code
train_loader_A = DataLoader(dataset_A, batch_size=batch_size, shuffle=True)

# 3. Model for A | W (Input: 2D, Output: 1D Gaussian parameters)
class A_given_WZ(nn.Module):
    def __init__(self):
        super().__init__()
        # Input dimension is now 2 because W has two components
        self.base = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU()
        )
        # scalar mean head for A
        self.mean_net = nn.Linear(8, 1)
        # scalar std head for A (softplus to ensure > 0)
        self.std_net  = nn.Sequential(
            nn.Linear(8, 1),
            nn.Softplus()
        )

    def forward(self, wz):
        h = self.base(wz)
        mu  = self.mean_net(h)
        std = self.std_net(h) + 1e-6 # add epsilon for stability
        return mu, std

# 4. Training Loop
model_A   = A_given_WZ().to(device)

opt_A = optim.AdamW(model_A.parameters(), lr=0.001)

num_epochs = 1000
best_val_nll = float('inf')

if train_all:
    print("Starting training for p(A|WZ)...")
    for epoch in range(1, num_epochs+1):
        # — Train —
        model_A.train()
        train_nll = 0.0
        for w_batch, a_true in train_loader_A:
            w_batch, a_true = w_batch.to(device), a_true.to(device)
            opt_A.zero_grad()
            mu, std = model_A(w_batch)
            dist = Normal(mu, std)
            loss = -dist.log_prob(a_true).mean()  # NLL
            loss.backward()
            opt_A.step()
            train_nll += loss.item()

        avg_train = train_nll / len(train_loader_A)
        
        if epoch % 50 == 0 or epoch == 1:
            print(f"[Epoch {epoch}/{num_epochs}] Train NLL: {avg_train:.4f}")

    torch.save(model_A.state_dict(), f'lowD_A_given_WZ_{sample_size}_{seed}.pth')

In [ ]:
set_seed(seed)

# 2. Define the Treatment Bridge Model q(Z, A)
class Q_BRIDGE(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: Z (2D) + A (1D) = 3
        self.fc = nn.Sequential(
            nn.Linear(2 + 1, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 1), # Outputs a scalar weight
            nn.Softplus()  # Ensure positivity
        )

    def forward(self, z, a):
        return self.fc(torch.cat([z, a], dim=1))

q_bridge = Q_BRIDGE().to(device)
optimizer = optim.AdamW(q_bridge.parameters(), lr=1e-2, weight_decay=1e-4)

K = 20

anchors = torch.linspace(A_tensor.min().item(), A_tensor.max().item(), K).to(device).unsqueeze(1)  # Shape [K, 1]

dataset = TensorDataset(Z_tensor, W_tensor)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

num_epochs = 1000
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)
best_train_loss = float('inf')

# --- 1. Load Pre-trained Components ---
# model_pA_WZ: p(A | W, Z) - Used for the MOMENT term
model_pA_WZ = A_given_WZ().to(device)
model_pA_WZ.load_state_dict(torch.load(f'lowD_A_given_WZ_{sample_size}_{seed}.pth', map_location=device))
model_pA_WZ.eval()

# model_h: Outcome Bridge h(W, A)
# model_h = NN_TW().to(device) 
model_h.load_state_dict(torch.load(f'lowD2_nn_tw_{sample_size}_{seed}_adv.pth', map_location=device))
model_h.eval()

lambd = 1e-3


# --- 4. Updated Training Loop with Automated Balancing ---
if True: 
    print(f"Starting Vectorized Training for q(Z,A) via Eq. 15 with Adaptive Balancing (N={sample_size})...")
    K_val = anchors.size(0)

    # Adaptive Tracking Variables
    running_moment_mag = 0.0
    running_norm_mag = 0.0
    alpha = 0.9 # Smoothing factor for the magnitude estimates

    for epoch in range(num_epochs):
        q_bridge.train()
        epoch_loss = 0.0
        
        for z_batch, w_batch in train_loader:
            B = z_batch.size(0)
            z_batch, w_batch = z_batch.to(device), w_batch.to(device)
            
            # --- Step 1: Create Augmented Tensors ---
            z_rep = z_batch.repeat(K_val, 1)
            w_rep = w_batch.repeat(K_val, 1)
            a_rep = anchors.repeat_interleave(B, dim=0)

            # --- Step 2: Vectorized Forward Passes ---
            with torch.no_grad():
                # 1. Outcome Bridge h(ak, wi)
                h_all = model_h(a_rep, w_rep).view(K_val, B) 
                unweighted_h = torch.mean(h_all, dim=1) 
                
                # 2. High-Res Density p(A=ak | zi, wi)
                WZ_conds = torch.cat([w_rep, z_rep], dim=1) 
                mu_awz, std_awz = model_pA_WZ(WZ_conds)
                dens_wz = torch.exp(Normal(mu_awz, std_awz).log_prob(a_rep)).view(K_val, B)

            # 3. Predict q
            q_all = q_bridge(z_rep, a_rep).view(K_val, B)
            wq = dens_wz * q_all

            # --- Step 3: Compute Individual Loss Components ---
            # Moment Matching (Signal Part)
            weighted_h = torch.mean(wq * h_all, dim=1) 
            loss_moment = torch.mean((weighted_h - unweighted_h)**2)

            # Normalization Constraint (Scale Part)
            norm_means = torch.mean(wq, dim=1)
            loss_norm = torch.mean((norm_means - 1.0)**2)

            # --- Step 4: ADAPTIVE WEIGHT CALCULATION ---
            with torch.no_grad():
                # Initialize or update the running average of magnitudes
                if running_moment_mag == 0:
                    running_moment_mag = loss_moment.item()
                    running_norm_mag = loss_norm.item()
                else:
                    running_moment_mag = alpha * running_moment_mag + (1-alpha) * loss_moment.item()
                    running_norm_mag = alpha * running_norm_mag + (1-alpha) * loss_norm.item()

                # Determine lambda to force a 50/50 balance of contributions
                # Magnitude(Moment) == lambd * Magnitude(Norm)
                lambd_auto = 0.25 * running_moment_mag / (running_norm_mag + 1e-8)
                
                # Sane clipping for synthetic stability
                lambd_auto = np.clip(lambd_auto, 1e-3, 1e3) 

            # Final Combined Loss
            loss = loss_moment + lambd_auto * loss_norm

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        scheduler.step()

        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1:03d}/{num_epochs}] | Total: {avg_loss:.6f} | Ratio: {lambd_auto:.4e} | Mom: {loss_moment.item():.6f} | Norm: {loss_norm.item():.6f}')

        if avg_loss < best_train_loss:
            best_train_loss = avg_loss
            torch.save(q_bridge.state_dict(), f'lowD_q_bridge_Eq178_{seed}.pth')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset, random_split

set_seed(seed)

# --- 1. Data Preparation and Standardization ---
Z_tensor = torch.tensor(data['Z'], dtype=torch.float32)               # [N, 2]
A_tensor = torch.tensor(data['A'], dtype=torch.float32).unsqueeze(1) # [N, 1]
Y_raw    = data['Y'].reshape(-1, 1)                                  # [N, 1]

Y_scaled = torch.tensor(Y_raw, dtype=torch.float32)

dataset_Y_given_ZA = TensorDataset(Z_tensor, A_tensor, Y_scaled)

train_loader = DataLoader(dataset_Y_given_ZA, batch_size=batch_size, shuffle=True)

# --- 1. Define the Outcome Model p(Y | Z, A) ---
class Y_given_ZA(nn.Module):
    def __init__(self, in_dim=3, out_dim=1):
        super().__init__()
        # Input: Z (2D) + A (1D) = 3D
        self.base = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, out_dim)
        )

    def forward(self, z, a):
        # Forward pass without U
        x = torch.cat([z, a], dim=1)
        return self.base(x)

# Instantiate
model_Y = Y_given_ZA(in_dim=3).to(device)

# Adjusted LR: 1e-3 is significantly more stable than 1e-1
optimizer = optim.AdamW(model_Y.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()
best_val_mse = float('inf')

num_epochs = 1000
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

if train_all:
    print("Starting MSE training for Y_given_ZA (No U)...")
    for epoch in range(1, num_epochs + 1):
        model_Y.train()
        epoch_train_mse = 0.0
        for Zb, Ab, Yb in train_loader:
            Zb, Ab, Yb = Zb.to(device), Ab.to(device), Yb.to(device)
            
            optimizer.zero_grad()
            # Predict Y directly from observed Z and A
            pred_y = model_Y(Zb, Ab)
            loss = criterion(pred_y, Yb)
            loss.backward()
            optimizer.step()
            epoch_train_mse += loss.item()

        avg_train_mse = epoch_train_mse / len(train_loader)
        scheduler.step()

        if epoch % 100 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Train MSE: {avg_train_mse:.6f}")

        if avg_train_mse < best_val_mse:
            best_val_mse = avg_train_mse
            torch.save(model_Y.state_dict(), f'lowD_Y_given_ZA_noU_{sample_size}_{seed}.pth')

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# 1. Load All Necessary Models
set_seed(seed)
model_Y = Y_given_ZA(in_dim=3).to(device)
model_Y.load_state_dict(torch.load(f'lowD_Y_given_ZA_noU_{sample_size}_{seed}.pth', map_location=device))

# We still need the q_bridge (or the encoder + density models) for the causal weights
q_bridge = Q_BRIDGE().to(device) # Your learned q(Z, a) from Eq 178
q_bridge.load_state_dict(torch.load(f'lowD_q_bridge_Eq178_{seed}.pth', map_location=device))

model_pA_WZ = A_given_WZ().to(device)
model_pA_WZ.load_state_dict(torch.load(f'lowD_A_given_WZ_{sample_size}_{seed}.pth', map_location=device))
model_pA_WZ.eval()

model_Y.eval(); q_bridge.eval(); model_pA_WZ.eval(); 

# 2. Parameters
Z_obs = Z_tensor.to(device)
W_obs = W_tensor.to(device)
WZ = torch.cat([W_obs, Z_obs], dim=1) 

N = Z_obs.size(0)
a_grid = np.linspace(-1.0, 2.0, 100) 
results_pred = []
results_true = []

mse_sum = 0.0
# 3. Evaluation Loop
for i, a in enumerate(a_grid):
    # Get ground truth
    ate_true = get_synthetic_ground_truth(a)
    results_true.append(ate_true)
    
    a_val = float(a)
    a_batch = torch.full((N, 1), a_val, device=device)

    with torch.no_grad():
        # ---- Step A: Calculate Causal Weights w = p(a|Z) * q(Z, a) ----
        # mu_p, std_p = pA_Z(Z_obs)
        mu_p, std_p = model_pA_WZ(WZ)

        p_density = torch.exp(Normal(mu_p, std_p).log_prob(a_batch)) 
        q_val = q_bridge(Z_obs, a_batch)
        weights = p_density * q_val # The weight for each individual

        # ---- Step B: Predict Outcome using the No-U Model ----
        # mu_y_z is the predicted outcome for each person at treatment a
        mu_y_z = model_Y(Z_obs, a_batch) # [N, 1]

        # ---- Step C: Weighted Average (Hajek Estimator) ----
        weighted_outcomes = (weights * mu_y_z).sum().item()
        total_weight = weights.sum().item()
        
        # This Hajek normalization is crucial for continuous treatment stability
        # ate_pred = weighted_outcomes / (total_weight + 1e-8)
        ate_pred = (weights * mu_y_z).mean().item()
        results_pred.append(ate_pred)
        mse_cur = (ate_pred - ate_true) ** 2
        mse_sum += mse_cur

    print(f"{a_val:6.2f} | {ate_true:10.6f} | {ate_pred:10.6f} | {mse_cur:10.8f}")

# 4. Visualization
plt.figure(figsize=(10, 6))
plt.plot(a_grid, results_true, 'k--', lw=2, label='True ATE (Target)')
plt.plot(a_grid, results_pred, 'r-', lw=2, label='ATE Pred')
plt.xlabel('Treatment A')
plt.ylabel('Outcome Y')
plt.title(f'Treatment Bridge Inference')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Average MSE: {mse_sum / len(a_grid):.8f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.distributions import Normal

set_seed(seed)

# 1. Load the Model for p(A | W, Z)
model_pA_WZ = A_given_WZ().to(device)
model_pA_WZ.load_state_dict(torch.load(f'lowD_A_given_WZ_{sample_size}_{seed}.pth', map_location=device))
model_pA_WZ.eval()

model_Y.eval(); q_bridge.eval(); nn_tw.eval()

# 2. Parameters
Z_obs = Z_tensor.to(device)
W_obs = W_tensor.to(device)
N = Z_obs.size(0)
M_bridge = 100 # Increased for a smoother conditional expectation

a_grid = np.linspace(-1.0, 2.0, 100)
results_dr = []
results_true = []
mse_sum = 0.0

print(f"Doubly Robust Evaluation (Equation 113 Style)")
print(f"{'a':>6} | {'True ATE':>10} | {'DR Pred':>10} | {'MSE':>10}")
print("-" * 55)

# 2. Evaluation Loop
for i, a in enumerate(a_grid):
    ate_true = get_synthetic_ground_truth(a)
    results_true.append(ate_true)
    
    a_val = float(a)
    a_batch = torch.full((N, 1), a_val, device=device)

    with torch.no_grad():
        # ---- Step A: Calculate Weights using p(A | W, Z) ----
        # Combine observed W and Z to find the propensity for current observation i
        WZ_obs_conds = torch.cat([W_obs, Z_obs], dim=1) # [N, 4]
        mu_p, std_p = model_pA_WZ(WZ_obs_conds)
        
        # Numerator: p(A=a | W, Z)
        p_density_wz = torch.exp(Normal(mu_p, std_p).log_prob(a_batch)) 
        
        # Bridge: q(Z, a)
        q_val = q_bridge(Z_obs, a_batch)
        
        # The accurate composite weight for Method 3 / Eq 178 logic
        weights = p_density_wz * q_val # [N, 1]

        # ---- Step B: Baseline OB (Mean of h on observed W) ----
        h_base = model_h(a_batch, W_obs)
        baseline_ob = h_base.mean().item()

        # ---- Step C: The Residual (mu - h) ----
        mu_y = model_Y(Z_obs, a_batch)
        residual = mu_y - h_base

        # ---- Step D: Doubly Robust Correction (Hajek style) ----
        # Using the p(A|WZ) weight here aligns with the observed W in h_base
        weighted_residual = (weights * residual).sum().item()
        total_weight = weights.sum().item()
        
        # correction = weighted_residual / (total_weight + 1e-8)
        correction = (weights * residual).mean().item()
        
        ate_pred = baseline_ob + correction
        results_dr.append(ate_pred)

    mse_cur = (ate_pred - ate_true)**2
    mse_sum += mse_cur
    if (i+1) % 20 == 0:
        print(f"{a_val:6.2f} | {ate_true:10.6f} | {ate_pred:10.6f} | {mse_cur:10.8f}")

print("-" * 55)
print(f"Average DR MSE: {mse_sum/len(a_grid):.8f}")

# 4. Final Visualization
plt.figure(figsize=(10, 6))
plt.plot(a_grid, results_true, 'k--', lw=3, label='True ATE (do-calculus)')
plt.plot(a_grid, results_dr, 'b-o', markersize=4, label='Doubly Robust Curve')
plt.plot(a_grid, ob_result_pred, 'r-', alpha=0.3, label='Outcome Bridge only')

plt.xlabel('Treatment A')
plt.ylabel('Outcome Y')
plt.title(f'Final Doubly Robust Evaluation (N={sample_size})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()